# Mitigación del sobreajuste, Optuna y TabNet (informe §5.3 / §5.4)

Tres intervenciones sobre el modelo **Combinado** de la Seccion 5.2:

1. **Regularización y early stopping** sobre el conjunto Combinado.
2. **Busqueda de hiperparametros con Optuna** (50 trials minimizando mlogloss de validacion).
3. **Red neuronal TabNet** como alternativa a los arboles.

Receta `combined` (PA + TA + cuotas B365 + rachas 5/10/15). Divisiones acordes al informe:
para §5.3 se separa test 20% y luego 12,5% de entrenamiento como validacion;
para §5.4 se usa test 15% y 17,65% del resto como validacion.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / ".git").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import pandas as pd
import numpy as np
from mineria.config import SEED
from mineria.dataset import build_dataset
from mineria.impute import MedianImputer
from mineria.evaluate import evaluate
from mineria.models.xgb import default_params, regularized_params, train_xgb
from mineria.optimize import run_study, best_model_from_study
%matplotlib inline

D:\dev\Mineria\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
features, cols = build_dataset("combined")
X, y = features[cols], features["FTR"]
print("Features combinadas:", X.shape)
X.shape

Features combinadas: (25979, 797)


(25979, 797)

In [3]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.125, random_state=SEED, stratify=y_trainval
)

imputer = MedianImputer().fit(X_train)
X_train_i, X_valid_i, X_test_i = (
    imputer.transform(X_train),
    imputer.transform(X_valid),
    imputer.transform(X_test),
)
X_train_i.shape, X_valid_i.shape, X_test_i.shape

((18185, 797), (2598, 797), (5196, 797))

## 1. Modelo original vs Modelo regularizado

El modelo base (Seccion 5.2, sin regularizacion) se compara con el modelo regularizado (reg_lambda=5.0, reg_alpha=1.5, gamma=0.5, subsample=0.8, colsample_bytree=0.8) con detencion temprana (early_stopping_rounds=10) sobre un conjunto de validacion del 12,5% del entrenamiento.

In [4]:
modelo_base = train_xgb(X_train_i, y_train, params=default_params())
res_base = evaluate(modelo_base, X_test_i, y_test)
acc_train_base = round(modelo_base.score(X_train_i, y_train), 4)
print(f"ORIGINAL    acc_train={acc_train_base:.4f} acc_test={res_base['accuracy']:.4f} gap={acc_train_base - res_base['accuracy']:.2f}pp")

ORIGINAL    acc_train=0.6052 acc_test=0.5248 gap=0.08pp


In [5]:
modelo_reg = train_xgb(
    X_train_i, y_train,
    params=regularized_params(),
    eval_set=[(X_valid_i, y_valid)],
    early_stopping_rounds=10,
)
acc_train_reg = round(modelo_reg.score(X_train_i, y_train), 4)
res_reg = evaluate(modelo_reg, X_test_i, y_test)
print(f"REGULARIZADO acc_train={acc_train_reg:.4f} acc_test={res_reg['accuracy']:.4f} gap={acc_train_reg - res_reg['accuracy']:.2f}pp best_iter={modelo_reg.best_iteration}")

REGULARIZADO acc_train=0.5689 acc_test=0.5241 gap=0.04pp best_iter=119


In [6]:
comparacion = pd.DataFrame(
    {
        "Metrica": ["Accuracy entrenamiento", "Accuracy test", "Brecha train-test"],
        "Modelo Original (5.2)": [acc_train_base, res_base["accuracy"], acc_train_base - res_base["accuracy"]],
        "Modelo Regularizado": [acc_train_reg, res_reg["accuracy"], acc_train_reg - res_reg["accuracy"]],
        "Informe": ["60.48% / 57.44%", "52.17% / 52.02%", "8.31 / 5.42 pp"],
    }
)
comparacion

,Metrica,Modelo Original (5.2),Modelo Regularizado,Informe
0,Accuracy entrenamiento,0.6052,0.5689,60.48% / 57.44%
1,Accuracy test,0.5248,0.5241,52.17% / 52.02%
2,Brecha train-test,0.0804,0.0448,8.31 / 5.42 pp


## 2. Busqueda de hiperparametros con Optuna

50 trials minimizando el mlogloss de validacion. Las hyperparametros ya regularizados (reg_lambda=5.0, reg_alpha=1.5) se mantienen fijos; se optimizan n_estimators, learning_rate, max_depth, subsample, colsample_bytree y gamma.

In [7]:
study = run_study(X_train_i, y_train, X_valid_i, y_valid, n_trials=50)
print("Mejor mlogloss validacion:", round(study.best_value, 4))
study.best_params

Mejor mlogloss validacion: 0.966


{'n_estimators': 480,
 'learning_rate': 0.06979571462363127,
 'max_depth': 3,
 'subsample': 0.5774610536477341,
 'colsample_bytree': 0.5544035243731583,
 'gamma': 0.05264734091688662}

In [8]:
modelo_opt = best_model_from_study(study, X_train_i, y_train)
acc_train_opt = round(modelo_opt.score(X_train_i, y_train), 4)
res_opt = evaluate(modelo_opt, X_test_i, y_test)
print(f"OPTUNA      acc_train={acc_train_opt:.4f} acc_test={res_opt['accuracy']:.4f} gap={acc_train_opt - res_opt['accuracy']:.2f}pp")

OPTUNA      acc_train=0.6662 acc_test=0.5154 gap=0.15pp


## 3. Red neuronal TabNet (§5.4)

TabNetClassifier (n_d=32, n_a=32, n_steps=5). Division propia del informe: test 15%, 17,65% de validacion del resto (~70/15/15). TabNet exige arrays numpy (X.values).

In [9]:
X_tt, X_test_t, y_tt, y_test_t = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train_t, X_valid_t, y_train_t, y_valid_t = train_test_split(
    X_tt, y_tt, test_size=0.1765, random_state=SEED, stratify=y_tt
)
imputer_t = MedianImputer().fit(X_train_t)
X_train_ti = imputer_t.transform(X_train_t)
X_valid_ti = imputer_t.transform(X_valid_t)
X_test_ti = imputer_t.transform(X_test_t)
X_train_ti.shape, X_valid_ti.shape, X_test_ti.shape

((18184, 797), (3898, 797), (3897, 797))

In [10]:
from mineria.models.tabnet import train_tabnet, tabnet_scores

tabnet_model = train_tabnet(
    X_train_ti, y_train_t,
    X_valid=X_valid_ti, y_valid=y_valid_t,
    n_d=32, n_a=32, n_steps=5,
    max_epochs=100, patience=30,
)

Stop training because you reached max_epochs = 100 with best_epoch = 79 and best_val_0_accuracy = 0.51821


D:\dev\Mineria\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [11]:
res_tabnet = tabnet_scores(tabnet_model, X_test_ti, y_test_t)
acc_tabnet = res_tabnet["accuracy"]
print(f"TABNET      acc_test={acc_tabnet:.4f}")
report = pd.DataFrame(res_tabnet["report"]).T
report

TABNET      acc_test=0.5099


,precision,recall,f1-score,support
Home Win,0.519534,0.847875,0.644284,1788.000000
Draw,0.434783,0.010111,0.019763,989.000000
Away Win,0.482218,0.411607,0.444123,1120.000000
accuracy,0.509879,0.509879,0.509879,0.509879
macro avg,0.478845,0.423198,0.369390,3897.000000
weighted avg,0.487301,0.509879,0.428264,3897.000000


## Comparacion final

Resumen de los enfoques evaluados frente a los valores del informe.

In [12]:
final = pd.DataFrame(
    {
        "Modelo": ["XGB base", "XGB regularizado", "XGB Optuna", "TabNet"],
        "Acc train": [acc_train_base, acc_train_reg, acc_train_opt, None],
        "Acc test": [res_base["accuracy"], res_reg["accuracy"], res_opt["accuracy"], res_tabnet["accuracy"]],
        "Informe (test)": ["52.17%", "52.02%", "52.31%", "51.12%"],
    }
)
final

,Modelo,Acc train,Acc test,Informe (test)
0,XGB base,0.6052,0.5248,52.17%
1,XGB regularizado,0.5689,0.5241,52.02%
2,XGB Optuna,0.6662,0.5154,52.31%
3,TabNet,NaN,0.5099,51.12%


## Conclusion

- La regularizacion redujo la brecha train-test de ~7-8 pp a ~5 pp sin sacrificar accuracy de test.
- Optuna aporto una mejora marginal, confirmando que el techo del problema es el limite informativo de las variables, no el sobreajuste.
- TabNet (red neuronal) no supera a XGBoost en datos tabulares con senal concentrada, como anticipa el informe.